In [2]:
# ============================================================
# PROJECT 01 - SRI LANKA TOURISM ANALYTICS
# NOTEBOOK 05 - STATISTICAL ANALYSIS
# ============================================================
#
# Analysis included:
#   1. Yearly growth
#   2. January-October comparable growth
#   3. Monthly descriptive statistics
#   4. Tourism volatility
#   5. Country contribution
#   6. Country growth
#   7. Top growing markets
#   8. Country correlation
#   9. Linear trend analysis
#  10. Statistical summary
#
# IMPORTANT:
# 2026 = January-October only.
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

from scipy import stats


# ============================================================
# 2. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

STATS_DIR = PROCESSED_DIR / "statistics"

STATS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. LOAD CLEAN DATA
# ============================================================

input_file = (
    PROCESSED_DIR /
    "tourism_arrivals_clean.csv"
)

tourism = pd.read_csv(input_file)


# ============================================================
# 4. DATA PREPARATION
# ============================================================

tourism["Date"] = pd.to_datetime(
    tourism["Date"],
    errors="coerce"
)

tourism["Year"] = pd.to_numeric(
    tourism["Year"],
    errors="coerce"
).astype("Int64")

tourism["Month_Number"] = pd.to_numeric(
    tourism["Month_Number"],
    errors="coerce"
).astype("Int64")

tourism["Arrivals"] = pd.to_numeric(
    tourism["Arrivals"],
    errors="coerce"
)

tourism["Country_Display"] = (
    tourism["Country_Standardized"]
    .fillna(tourism["Country_Raw"])
    .astype(str)
    .str.strip()
)


# ============================================================
# 5. BASIC INFORMATION
# ============================================================

print("=" * 75)
print("SRI LANKA TOURISM - STATISTICAL ANALYSIS")
print("=" * 75)

print("\nDataset loaded successfully.")

print("Rows:", len(tourism))

print(
    "Countries:",
    tourism["Country_Display"].nunique()
)

print(
    "Years:",
    sorted(
        tourism["Year"]
        .dropna()
        .unique()
        .tolist()
    )
)

print(
    "Missing arrivals:",
    tourism["Arrivals"].isna().sum()
)

print("\n2026 is a partial year: January-October.")


# ============================================================
# 6. YEARLY ARRIVALS
# ============================================================

yearly = (
    tourism
    .groupby(
        "Year",
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values("Year")
)


# ============================================================
# 7. YEAR-OVER-YEAR GROWTH
# ============================================================

yearly["YoY_Growth_%"] = (
    yearly["Arrivals"]
    .pct_change()
    * 100
)

# 2026 is partial, so don't use its full-year YoY
yearly.loc[
    yearly["Year"] == 2026,
    "YoY_Growth_%"
] = np.nan


print("\n")
print("=" * 75)
print("1. YEARLY GROWTH ANALYSIS")
print("=" * 75)

print(
    yearly.to_string(
        index=False
    )
)


# ============================================================
# 8. JANUARY-OCTOBER COMPARABLE DATA
# ============================================================

jan_oct = (
    tourism[
        tourism["Month_Number"] <= 10
    ]
    .groupby(
        "Year",
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values("Year")
)

jan_oct["YoY_Growth_%"] = (
    jan_oct["Arrivals"]
    .pct_change()
    * 100
)


print("\n")
print("=" * 75)
print("2. JANUARY-OCTOBER COMPARABLE GROWTH")
print("=" * 75)

print(
    jan_oct.to_string(
        index=False
    )
)


# ============================================================
# 9. MONTHLY TOURISM STATISTICS
# ============================================================

monthly = (
    tourism
    .groupby(
        ["Date", "Year", "Month_Number", "Month"],
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values("Date")
)

monthly["Monthly_Growth_%"] = (
    monthly["Arrivals"]
    .pct_change()
    * 100
)


# ============================================================
# 10. DESCRIPTIVE STATISTICS
# ============================================================

monthly_stats = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Standard Deviation",
        "Variance",
        "Coefficient of Variation (%)"
    ],
    "Value": [
        monthly["Arrivals"].mean(),
        monthly["Arrivals"].median(),
        monthly["Arrivals"].min(),
        monthly["Arrivals"].max(),
        monthly["Arrivals"].std(),
        monthly["Arrivals"].var(),
        (
            monthly["Arrivals"].std()
            / monthly["Arrivals"].mean()
            * 100
        )
    ]
})


print("\n")
print("=" * 75)
print("3. MONTHLY DESCRIPTIVE STATISTICS")
print("=" * 75)

print(
    monthly_stats.to_string(
        index=False
    )
)


# ============================================================
# 11. BEST AND WORST MONTHS
# ============================================================

best_month = monthly.loc[
    monthly["Arrivals"].idxmax()
]

worst_month = monthly.loc[
    monthly["Arrivals"].idxmin()
]


print("\n")
print("=" * 75)
print("4. BEST AND WORST MONTHS")
print("=" * 75)

print(
    "Best month:",
    best_month["Month"],
    int(best_month["Year"]),
    "-",
    f"{best_month['Arrivals']:,.0f}",
    "arrivals"
)

print(
    "Lowest month:",
    worst_month["Month"],
    int(worst_month["Year"]),
    "-",
    f"{worst_month['Arrivals']:,.0f}",
    "arrivals"
)


# ============================================================
# 12. MONTHLY SEASONAL STATISTICS
# ============================================================

seasonality = (
    tourism
    .groupby(
        ["Month_Number", "Month"],
        as_index=False
    )["Arrivals"]
    .agg(
        Average="mean",
        Median="median",
        Minimum="min",
        Maximum="max",
        Standard_Deviation="std"
    )
    .sort_values("Month_Number")
)

seasonality["Coefficient_of_Variation_%"] = (
    seasonality["Standard_Deviation"]
    / seasonality["Average"]
    * 100
)


print("\n")
print("=" * 75)
print("5. SEASONAL STATISTICS")
print("=" * 75)

print(
    seasonality.to_string(
        index=False
    )
)


# ============================================================
# 13. COUNTRY TOTALS
# ============================================================

country_totals = (
    tourism
    .groupby(
        "Country_Display",
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values(
        "Arrivals",
        ascending=False
    )
)

total_arrivals = (
    country_totals["Arrivals"].sum()
)

country_totals["Share_%"] = (
    country_totals["Arrivals"]
    / total_arrivals
    * 100
)


# ============================================================
# 14. COUNTRY CONTRIBUTION
# ============================================================

print("\n")
print("=" * 75)
print("6. COUNTRY CONTRIBUTION")
print("=" * 75)

print(
    country_totals
    .head(15)
    .to_string(index=False)
)


# ============================================================
# 15. COUNTRY DATA BY YEAR
# ============================================================

country_year = (
    tourism
    .groupby(
        ["Country_Display", "Year"],
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
)


country_year_pivot = (
    country_year
    .pivot(
        index="Country_Display",
        columns="Year",
        values="Arrivals"
    )
)


# ============================================================
# 16. COUNTRY GROWTH 2022 TO 2025
# ============================================================
#
# We use 2022 and 2025 because both are complete years.
# 2026 is excluded from this full-year comparison.
# ============================================================

country_growth = (
    country_year_pivot
    .copy()
)

if 2022 in country_growth.columns and 2025 in country_growth.columns:

    country_growth["Growth_2022_to_2025_%"] = np.where(
        country_growth[2022] > 0,
        (
            (
                country_growth[2025]
                -
                country_growth[2022]
            )
            /
            country_growth[2022]
            * 100
        ),
        np.nan
    )

    country_growth["Absolute_Change"] = (
        country_growth[2025]
        -
        country_growth[2022]
    )

    country_growth = (
        country_growth
        .reset_index()
        .sort_values(
            "Growth_2022_to_2025_%",
            ascending=False
        )
    )

else:

    country_growth = pd.DataFrame()


# ============================================================
# 17. TOP GROWING MARKETS
# ============================================================

if not country_growth.empty:

    # Require at least 10,000 arrivals in 2022
    # to avoid tiny markets producing misleading
    # extreme growth percentages.

    top_growth = (
        country_growth[
            country_growth[2022] >= 10000
        ]
        .sort_values(
            "Growth_2022_to_2025_%",
            ascending=False
        )
        .head(15)
    )

    declining_markets = (
        country_growth[
            country_growth[2022] >= 10000
        ]
        .sort_values(
            "Growth_2022_to_2025_%"
        )
        .head(15)
    )

else:

    top_growth = pd.DataFrame()

    declining_markets = pd.DataFrame()


print("\n")
print("=" * 75)
print("7. FASTEST-GROWING SOURCE MARKETS")
print("=" * 75)

if not top_growth.empty:

    print(
        top_growth[
            [
                "Country_Display",
                2022,
                2025,
                "Growth_2022_to_2025_%",
                "Absolute_Change"
            ]
        ].to_string(index=False)
    )


print("\n")
print("=" * 75)
print("8. DECLINING / SLOWEST-GROWING MARKETS")
print("=" * 75)

if not declining_markets.empty:

    print(
        declining_markets[
            [
                "Country_Display",
                2022,
                2025,
                "Growth_2022_to_2025_%",
                "Absolute_Change"
            ]
        ].to_string(index=False)
    )


# ============================================================
# 18. TOP COUNTRIES - 2026 JAN-OCT
# ============================================================

country_2026 = (
    tourism[
        (tourism["Year"] == 2026)
        &
        (tourism["Month_Number"] <= 10)
    ]
    .groupby(
        "Country_Display",
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values(
        "Arrivals",
        ascending=False
    )
)

country_2026["Share_%"] = (
    country_2026["Arrivals"]
    /
    country_2026["Arrivals"].sum()
    * 100
)


print("\n")
print("=" * 75)
print("9. TOP SOURCE MARKETS - 2026 JAN-OCT")
print("=" * 75)

print(
    country_2026
    .head(15)
    .to_string(index=False)
)


# ============================================================
# 19. TOP COUNTRY CORRELATION ANALYSIS
# ============================================================
#
# Correlation shows whether major source markets tend to
# rise and fall together across the monthly time series.
# ============================================================

top10_names = (
    country_totals
    .head(10)["Country_Display"]
    .tolist()
)

country_monthly = (
    tourism[
        tourism["Country_Display"]
        .isin(top10_names)
    ]
    .groupby(
        ["Date", "Country_Display"],
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
)

country_monthly_pivot = (
    country_monthly
    .pivot(
        index="Date",
        columns="Country_Display",
        values="Arrivals"
    )
)

correlation_matrix = (
    country_monthly_pivot
    .corr()
)


print("\n")
print("=" * 75)
print("10. TOP 10 SOURCE MARKET CORRELATION")
print("=" * 75)

print(
    correlation_matrix.round(2).to_string()
)


# ============================================================
# 20. STRONGEST COUNTRY CORRELATION PAIR
# ============================================================

corr_pairs = (
    correlation_matrix
    .where(
        np.triu(
            np.ones(
                correlation_matrix.shape
            ),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

corr_pairs.columns = [
    "Country_1",
    "Country_2",
    "Correlation"
]

corr_pairs = corr_pairs.sort_values(
    "Correlation",
    ascending=False
)


if len(corr_pairs) > 0:

    strongest_pair = corr_pairs.iloc[0]

    weakest_pair = corr_pairs.iloc[-1]

    print("\nStrongest positive correlation:")

    print(
        strongest_pair["Country_1"],
        "&",
        strongest_pair["Country_2"],
        "=",
        round(
            strongest_pair["Correlation"],
            3
        )
    )

    print("\nWeakest correlation:")

    print(
        weakest_pair["Country_1"],
        "&",
        weakest_pair["Country_2"],
        "=",
        round(
            weakest_pair["Correlation"],
            3
        )
    )


# ============================================================
# 21. LINEAR TREND ANALYSIS
# ============================================================
#
# Full-year trend uses 2022-2025 only because 2026
# is incomplete.
# ============================================================

full_years = yearly[
    yearly["Year"] <= 2025
].copy()

trend_result = stats.linregress(
    full_years["Year"].astype(float),
    full_years["Arrivals"].astype(float)
)


trend_summary = pd.DataFrame({
    "Metric": [
        "Slope",
        "Intercept",
        "R-squared",
        "P-value",
        "Standard Error"
    ],
    "Value": [
        trend_result.slope,
        trend_result.intercept,
        trend_result.rvalue ** 2,
        trend_result.pvalue,
        trend_result.stderr
    ]
})


print("\n")
print("=" * 75)
print("11. FULL-YEAR LINEAR TREND ANALYSIS")
print("=" * 75)

print(
    trend_summary.to_string(
        index=False
    )
)


# ============================================================
# 22. TREND INTERPRETATION
# ============================================================

print("\nTrend interpretation:")

if trend_result.slope > 0:
    print(
        "The full-year trend from 2022-2025 is upward."
    )
else:
    print(
        "The full-year trend from 2022-2025 is downward."
    )

print(
    "R-squared:",
    round(
        trend_result.rvalue ** 2,
        4
    )
)

print(
    "P-value:",
    round(
        trend_result.pvalue,
        4
    )
)


# ============================================================
# 23. YEARLY VOLATILITY
# ============================================================

full_year_arrivals = (
    yearly[
        yearly["Year"] <= 2025
    ]["Arrivals"]
)

yearly_volatility = pd.DataFrame({
    "Metric": [
        "Mean Full-Year Arrivals",
        "Median Full-Year Arrivals",
        "Standard Deviation",
        "Coefficient of Variation (%)",
        "Minimum",
        "Maximum"
    ],
    "Value": [
        full_year_arrivals.mean(),
        full_year_arrivals.median(),
        full_year_arrivals.std(),
        (
            full_year_arrivals.std()
            /
            full_year_arrivals.mean()
            * 100
        ),
        full_year_arrivals.min(),
        full_year_arrivals.max()
    ]
})


print("\n")
print("=" * 75)
print("12. FULL-YEAR VOLATILITY")
print("=" * 75)

print(
    yearly_volatility.to_string(
        index=False
    )
)


# ============================================================
# 24. SAVE STATISTICAL OUTPUTS
# ============================================================

yearly.to_csv(
    STATS_DIR / "yearly_growth.csv",
    index=False
)

jan_oct.to_csv(
    STATS_DIR / "jan_oct_growth.csv",
    index=False
)

monthly_stats.to_csv(
    STATS_DIR / "monthly_descriptive_statistics.csv",
    index=False
)

seasonality.to_csv(
    STATS_DIR / "seasonal_statistics.csv",
    index=False
)

country_totals.to_csv(
    STATS_DIR / "country_contribution.csv",
    index=False
)

country_growth.to_csv(
    STATS_DIR / "country_growth_2022_2025.csv",
    index=False
)

country_2026.to_csv(
    STATS_DIR / "country_2026_jan_oct.csv",
    index=False
)

correlation_matrix.to_csv(
    STATS_DIR / "country_correlation_matrix.csv"
)

corr_pairs.to_csv(
    STATS_DIR / "country_correlation_pairs.csv",
    index=False
)

trend_summary.to_csv(
    STATS_DIR / "linear_trend_analysis.csv",
    index=False
)

yearly_volatility.to_csv(
    STATS_DIR / "yearly_volatility.csv",
    index=False
)


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 75)
print("STATISTICAL ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 75)

print("\nStatistical output folder:")
print(STATS_DIR)

print("\nFiles created:")

print("  yearly_growth.csv")
print("  jan_oct_growth.csv")
print("  monthly_descriptive_statistics.csv")
print("  seasonal_statistics.csv")
print("  country_contribution.csv")
print("  country_growth_2022_2025.csv")
print("  country_2026_jan_oct.csv")
print("  country_correlation_matrix.csv")
print("  country_correlation_pairs.csv")
print("  linear_trend_analysis.csv")
print("  yearly_volatility.csv")

print("\nImportant analytical rule:")
print(
    "2026 is January-October only and is not used "
    "as a complete-year comparison against 2022-2025."
)

print("\nNext stage after this:")
print("SQL Analysis + SQLite")

print("=" * 75)

SRI LANKA TOURISM - STATISTICAL ANALYSIS

Dataset loaded successfully.
Rows: 11110
Countries: 229
Years: [2022, 2023, 2024, 2025, 2026]
Missing arrivals: 2

2026 is a partial year: January-October.


1. YEARLY GROWTH ANALYSIS
 Year  Arrivals  YoY_Growth_%
 2022  719978.0           NaN
 2023 1487303.0    106.576173
 2024 2053465.0     38.066352
 2025 2362521.0     15.050463
 2026 1862304.0           NaN


2. JANUARY-OCTOBER COMPARABLE GROWTH
 Year  Arrivals  YoY_Growth_%
 2022  568258.0           NaN
 2023 1125455.0     98.053525
 2024 1620715.0     44.005313
 2025 1890687.0     16.657586
 2026 1862304.0     -1.501200


3. MONTHLY DESCRIPTIVE STATISTICS
                      Metric        Value
                        Mean 1.463029e+05
                      Median 1.406400e+05
                     Minimum 2.980200e+04
                     Maximum 2.793280e+05
          Standard Deviation 6.484077e+04
                    Variance 4.204325e+09
Coefficient of Variation (%) 4.431952e+01


4

ValueError: cannot insert Country_Display, already exists

In [4]:
# ============================================================
# PROJECT 01 - SRI LANKA TOURISM ANALYTICS
# NOTEBOOK 05 - STATISTICAL ANALYSIS
# CORRECTED COMPLETE VERSION
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
from scipy import stats


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

STATS_DIR = PROCESSED_DIR / "statistics"

STATS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. LOAD CLEAN DATA
# ============================================================

input_file = (
    PROCESSED_DIR /
    "tourism_arrivals_clean.csv"
)

tourism = pd.read_csv(input_file)


# ============================================================
# 3. DATA PREPARATION
# ============================================================

tourism["Date"] = pd.to_datetime(
    tourism["Date"],
    errors="coerce"
)

tourism["Year"] = pd.to_numeric(
    tourism["Year"],
    errors="coerce"
).astype("Int64")

tourism["Month_Number"] = pd.to_numeric(
    tourism["Month_Number"],
    errors="coerce"
).astype("Int64")

tourism["Arrivals"] = pd.to_numeric(
    tourism["Arrivals"],
    errors="coerce"
)

tourism["Country_Display"] = (
    tourism["Country_Standardized"]
    .fillna(tourism["Country_Raw"])
    .astype(str)
    .str.strip()
)


# ============================================================
# 4. BASIC INFORMATION
# ============================================================

print("=" * 75)
print("SRI LANKA TOURISM - STATISTICAL ANALYSIS")
print("=" * 75)

print("\nDataset loaded successfully.")
print("Rows:", len(tourism))
print(
    "Countries:",
    tourism["Country_Display"].nunique()
)
print(
    "Years:",
    sorted(
        tourism["Year"]
        .dropna()
        .unique()
        .tolist()
    )
)
print(
    "Missing arrivals:",
    tourism["Arrivals"].isna().sum()
)

print("\n2026 is a partial year: January-October.")


# ============================================================
# 5. YEARLY ARRIVALS
# ============================================================

yearly = (
    tourism
    .groupby(
        "Year",
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values("Year")
)

yearly["YoY_Growth_%"] = (
    yearly["Arrivals"]
    .pct_change()
    * 100
)

# 2026 is partial
yearly.loc[
    yearly["Year"] == 2026,
    "YoY_Growth_%"
] = np.nan


print("\n" + "=" * 75)
print("1. YEARLY GROWTH ANALYSIS")
print("=" * 75)

print(
    yearly.to_string(index=False)
)


# ============================================================
# 6. JANUARY-OCTOBER COMPARISON
# ============================================================

jan_oct = (
    tourism[
        tourism["Month_Number"] <= 10
    ]
    .groupby(
        "Year",
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values("Year")
)

jan_oct["YoY_Growth_%"] = (
    jan_oct["Arrivals"]
    .pct_change()
    * 100
)


print("\n" + "=" * 75)
print("2. JANUARY-OCTOBER COMPARABLE GROWTH")
print("=" * 75)

print(
    jan_oct.to_string(index=False)
)


# ============================================================
# 7. MONTHLY DATA
# ============================================================

monthly = (
    tourism
    .groupby(
        [
            "Date",
            "Year",
            "Month_Number",
            "Month"
        ],
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values("Date")
)

monthly["Monthly_Growth_%"] = (
    monthly["Arrivals"]
    .pct_change()
    * 100
)


# ============================================================
# 8. DESCRIPTIVE STATISTICS
# ============================================================

monthly_stats = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Standard Deviation",
        "Variance",
        "Coefficient of Variation (%)"
    ],
    "Value": [
        monthly["Arrivals"].mean(),
        monthly["Arrivals"].median(),
        monthly["Arrivals"].min(),
        monthly["Arrivals"].max(),
        monthly["Arrivals"].std(),
        monthly["Arrivals"].var(),
        (
            monthly["Arrivals"].std()
            /
            monthly["Arrivals"].mean()
            * 100
        )
    ]
})


print("\n" + "=" * 75)
print("3. MONTHLY DESCRIPTIVE STATISTICS")
print("=" * 75)

print(
    monthly_stats.to_string(index=False)
)


# ============================================================
# 9. BEST AND WORST MONTH
# ============================================================

best_month = monthly.loc[
    monthly["Arrivals"].idxmax()
]

worst_month = monthly.loc[
    monthly["Arrivals"].idxmin()
]

print("\n" + "=" * 75)
print("4. BEST AND WORST MONTHS")
print("=" * 75)

print(
    "Best month:",
    best_month["Month"],
    int(best_month["Year"]),
    "-",
    f"{best_month['Arrivals']:,.0f}",
    "arrivals"
)

print(
    "Lowest month:",
    worst_month["Month"],
    int(worst_month["Year"]),
    "-",
    f"{worst_month['Arrivals']:,.0f}",
    "arrivals"
)


# ============================================================
# 10. SEASONAL STATISTICS
# ============================================================

seasonality = (
    tourism
    .groupby(
        ["Month_Number", "Month"],
        as_index=False
    )["Arrivals"]
    .agg(
        Average="mean",
        Median="median",
        Minimum="min",
        Maximum="max",
        Standard_Deviation="std"
    )
    .sort_values("Month_Number")
)

seasonality["Coefficient_of_Variation_%"] = (
    seasonality["Standard_Deviation"]
    /
    seasonality["Average"]
    * 100
)


print("\n" + "=" * 75)
print("5. SEASONAL STATISTICS")
print("=" * 75)

print(
    seasonality.to_string(index=False)
)


# ============================================================
# 11. COUNTRY CONTRIBUTION
# ============================================================

country_totals = (
    tourism
    .groupby(
        "Country_Display",
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values(
        "Arrivals",
        ascending=False
    )
)

total_arrivals = (
    country_totals["Arrivals"].sum()
)

country_totals["Share_%"] = (
    country_totals["Arrivals"]
    /
    total_arrivals
    * 100
)


print("\n" + "=" * 75)
print("6. COUNTRY CONTRIBUTION - TOP 15")
print("=" * 75)

print(
    country_totals
    .head(15)
    .to_string(index=False)
)


# ============================================================
# 12. COUNTRY BY YEAR
# ============================================================

country_year = (
    tourism
    .groupby(
        [
            "Country_Display",
            "Year"
        ],
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
)

country_year_pivot = (
    country_year
    .pivot(
        index="Country_Display",
        columns="Year",
        values="Arrivals"
    )
)


# ============================================================
# 13. COUNTRY GROWTH 2022 TO 2025
# ============================================================

if 2022 in country_year_pivot.columns and 2025 in country_year_pivot.columns:

    country_growth = (
        country_year_pivot[
            [2022, 2025]
        ]
        .copy()
        .reset_index()
    )

    country_growth["Growth_2022_to_2025_%"] = np.where(
        country_growth[2022] > 0,
        (
            (
                country_growth[2025]
                -
                country_growth[2022]
            )
            /
            country_growth[2022]
            *
            100
        ),
        np.nan
    )

    country_growth["Absolute_Change"] = (
        country_growth[2025]
        -
        country_growth[2022]
    )

    country_growth = country_growth.sort_values(
        "Growth_2022_to_2025_%",
        ascending=False
    )

else:

    country_growth = pd.DataFrame()


# ============================================================
# 14. FASTEST-GROWING MARKETS
# ============================================================

if not country_growth.empty:

    top_growth = (
        country_growth[
            country_growth[2022] >= 10000
        ]
        .sort_values(
            "Growth_2022_to_2025_%",
            ascending=False
        )
        .head(15)
    )

else:

    top_growth = pd.DataFrame()


print("\n" + "=" * 75)
print("7. FASTEST-GROWING SOURCE MARKETS")
print("=" * 75)

if not top_growth.empty:

    print(
        top_growth[
            [
                "Country_Display",
                2022,
                2025,
                "Growth_2022_to_2025_%",
                "Absolute_Change"
            ]
        ].to_string(index=False)
    )


# ============================================================
# 15. SLOWEST / DECLINING MARKETS
# ============================================================

if not country_growth.empty:

    declining_markets = (
        country_growth[
            country_growth[2022] >= 10000
        ]
        .sort_values(
            "Growth_2022_to_2025_%"
        )
        .head(15)
    )

else:

    declining_markets = pd.DataFrame()


print("\n" + "=" * 75)
print("8. SLOWEST-GROWING / DECLINING MARKETS")
print("=" * 75)

if not declining_markets.empty:

    print(
        declining_markets[
            [
                "Country_Display",
                2022,
                2025,
                "Growth_2022_to_2025_%",
                "Absolute_Change"
            ]
        ].to_string(index=False)
    )


# ============================================================
# 16. 2026 JAN-OCT TOP MARKETS
# ============================================================

country_2026 = (
    tourism[
        (tourism["Year"] == 2026)
        &
        (tourism["Month_Number"] <= 10)
    ]
    .groupby(
        "Country_Display",
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values(
        "Arrivals",
        ascending=False
    )
)

country_2026["Share_%"] = (
    country_2026["Arrivals"]
    /
    country_2026["Arrivals"].sum()
    *
    100
)


print("\n" + "=" * 75)
print("9. TOP SOURCE MARKETS - 2026 JAN-OCT")
print("=" * 75)

print(
    country_2026
    .head(15)
    .to_string(index=False)
)


# ============================================================
# 17. TOP 10 COUNTRY MONTHLY DATA
# ============================================================

top10_names = (
    country_totals
    .head(10)["Country_Display"]
    .tolist()
)

country_monthly = (
    tourism[
        tourism["Country_Display"]
        .isin(top10_names)
    ]
    .groupby(
        [
            "Date",
            "Country_Display"
        ],
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
)

country_monthly_pivot = (
    country_monthly
    .pivot(
        index="Date",
        columns="Country_Display",
        values="Arrivals"
    )
)


# ============================================================
# 18. CORRELATION MATRIX
# ============================================================

correlation_matrix = (
    country_monthly_pivot.corr()
)


print("\n" + "=" * 75)
print("10. TOP 10 SOURCE MARKET CORRELATION")
print("=" * 75)

print(
    correlation_matrix
    .round(2)
    .to_string()
)


# ============================================================
# 19. CORRELATION PAIRS
# ============================================================
#
# FIXED VERSION
# ============================================================

corr_values = []

countries = (
    correlation_matrix.columns.tolist()
)

for i in range(len(countries)):

    for j in range(i + 1, len(countries)):

        country_1 = countries[i]

        country_2 = countries[j]

        correlation_value = (
            correlation_matrix.loc[
                country_1,
                country_2
            ]
        )

        corr_values.append({
            "Country_1": country_1,
            "Country_2": country_2,
            "Correlation": correlation_value
        })


corr_pairs = pd.DataFrame(
    corr_values
)

corr_pairs = corr_pairs.sort_values(
    "Correlation",
    ascending=False
)


# ============================================================
# 20. STRONGEST / WEAKEST CORRELATION
# ============================================================

print("\n" + "=" * 75)
print("11. CORRELATION PAIR ANALYSIS")
print("=" * 75)

if len(corr_pairs) > 0:

    strongest_pair = (
        corr_pairs.iloc[0]
    )

    weakest_pair = (
        corr_pairs.iloc[-1]
    )

    print(
        "Strongest positive correlation:"
    )

    print(
        strongest_pair["Country_1"],
        "&",
        strongest_pair["Country_2"],
        "=",
        round(
            strongest_pair["Correlation"],
            3
        )
    )

    print(
        "\nWeakest correlation:"
    )

    print(
        weakest_pair["Country_1"],
        "&",
        weakest_pair["Country_2"],
        "=",
        round(
            weakest_pair["Correlation"],
            3
        )
    )


# ============================================================
# 21. LINEAR TREND ANALYSIS
# ============================================================

full_years = (
    yearly[
        yearly["Year"] <= 2025
    ]
    .copy()
)

trend_result = stats.linregress(
    full_years["Year"].astype(float),
    full_years["Arrivals"].astype(float)
)

trend_summary = pd.DataFrame({
    "Metric": [
        "Slope",
        "Intercept",
        "R-squared",
        "P-value",
        "Standard Error"
    ],
    "Value": [
        trend_result.slope,
        trend_result.intercept,
        trend_result.rvalue ** 2,
        trend_result.pvalue,
        trend_result.stderr
    ]
})


print("\n" + "=" * 75)
print("12. FULL-YEAR LINEAR TREND ANALYSIS")
print("=" * 75)

print(
    trend_summary.to_string(
        index=False
    )
)

if trend_result.slope > 0:

    print(
        "\nTrend direction: UPWARD"
    )

else:

    print(
        "\nTrend direction: DOWNWARD"
    )


# ============================================================
# 22. FULL-YEAR VOLATILITY
# ============================================================

full_year_arrivals = (
    yearly[
        yearly["Year"] <= 2025
    ]["Arrivals"]
)

yearly_volatility = pd.DataFrame({
    "Metric": [
        "Mean Full-Year Arrivals",
        "Median Full-Year Arrivals",
        "Standard Deviation",
        "Coefficient of Variation (%)",
        "Minimum",
        "Maximum"
    ],
    "Value": [
        full_year_arrivals.mean(),
        full_year_arrivals.median(),
        full_year_arrivals.std(),
        (
            full_year_arrivals.std()
            /
            full_year_arrivals.mean()
            *
            100
        ),
        full_year_arrivals.min(),
        full_year_arrivals.max()
    ]
})


print("\n" + "=" * 75)
print("13. FULL-YEAR VOLATILITY")
print("=" * 75)

print(
    yearly_volatility.to_string(
        index=False
    )
)


# ============================================================
# 23. SAVE ALL RESULTS
# ============================================================

yearly.to_csv(
    STATS_DIR / "yearly_growth.csv",
    index=False
)

jan_oct.to_csv(
    STATS_DIR / "jan_oct_growth.csv",
    index=False
)

monthly_stats.to_csv(
    STATS_DIR /
    "monthly_descriptive_statistics.csv",
    index=False
)

seasonality.to_csv(
    STATS_DIR /
    "seasonal_statistics.csv",
    index=False
)

country_totals.to_csv(
    STATS_DIR /
    "country_contribution.csv",
    index=False
)

country_growth.to_csv(
    STATS_DIR /
    "country_growth_2022_2025.csv",
    index=False
)

country_2026.to_csv(
    STATS_DIR /
    "country_2026_jan_oct.csv",
    index=False
)

correlation_matrix.to_csv(
    STATS_DIR /
    "country_correlation_matrix.csv"
)

corr_pairs.to_csv(
    STATS_DIR /
    "country_correlation_pairs.csv",
    index=False
)

trend_summary.to_csv(
    STATS_DIR /
    "linear_trend_analysis.csv",
    index=False
)

yearly_volatility.to_csv(
    STATS_DIR /
    "yearly_volatility.csv",
    index=False
)


# ============================================================
# 24. FINAL REPORT
# ============================================================

print("\n")
print("=" * 75)
print("STATISTICAL ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 75)

print("\nStatistics folder:")
print(STATS_DIR)

print("\nFiles created:")

for file in sorted(STATS_DIR.glob("*.csv")):

    print(
        " ",
        file.name
    )

print(
    "\nTotal statistical output files:",
    len(list(STATS_DIR.glob("*.csv")))
)

print("\nImportant:")
print(
    "2026 = January-October only."
)

print(
    "2022-2025 are used for complete-year trend analysis."
)

print(
    "2026 Jan-Oct is used for comparable current-year analysis."
)

print("\nNEXT STAGE:")
print("SQL Analysis + SQLite")

print("=" * 75)

SRI LANKA TOURISM - STATISTICAL ANALYSIS

Dataset loaded successfully.
Rows: 11110
Countries: 229
Years: [2022, 2023, 2024, 2025, 2026]
Missing arrivals: 2

2026 is a partial year: January-October.

1. YEARLY GROWTH ANALYSIS
 Year  Arrivals  YoY_Growth_%
 2022  719978.0           NaN
 2023 1487303.0    106.576173
 2024 2053465.0     38.066352
 2025 2362521.0     15.050463
 2026 1862304.0           NaN

2. JANUARY-OCTOBER COMPARABLE GROWTH
 Year  Arrivals  YoY_Growth_%
 2022  568258.0           NaN
 2023 1125455.0     98.053525
 2024 1620715.0     44.005313
 2025 1890687.0     16.657586
 2026 1862304.0     -1.501200

3. MONTHLY DESCRIPTIVE STATISTICS
                      Metric        Value
                        Mean 1.463029e+05
                      Median 1.406400e+05
                     Minimum 2.980200e+04
                     Maximum 2.793280e+05
          Standard Deviation 6.484077e+04
                    Variance 4.204325e+09
Coefficient of Variation (%) 4.431952e+01

4. BE